In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# ==========================================
# STEP 1: LOAD & CLEAN DATA (UPDATED FIX)
# ==========================================
print("--- STEP 1: Loading Data ---")
df = pd.read_csv('pump_data (2).csv')

# NEW FIX: Force the sensor columns to be purely mathematical numbers.
# If it finds a word like 'NORMAL' in the Power column, it will turn it into an empty 'NaN'
feature_cols = ['Voltage', 'Current', 'Power']
for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Now drop any rows that have missing values (or the NaN values we just created from the corrupted rows)
df = df.dropna().drop_duplicates()
print(f"Shape after removing corrupted rows: {df.shape}")

# Prevent Data Leakage
if 'Time' in df.columns:
    df = df.drop(columns=['Time'])

# Encode the 3 Target Variables ('State')
label_encoder = LabelEncoder()
df['State'] = label_encoder.fit_transform(df['State'])
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print(f"Classes Found: {label_mapping}")

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

reverse_mapping = {v: k for k, v in label_mapping.items()}
df_plot = df.copy()
df_plot['State_Name'] = df_plot['State'].map(reverse_mapping)

features = ['Voltage', 'Current', 'Power']

# Plot Feature Distributions for the 3 classes
plt.figure(figsize=(15, 5))
for i, feature in enumerate(features, 1):
    plt.subplot(1, 3, i)
    sns.boxplot(data=df_plot, x='State_Name', y=feature, palette="Set2")
    plt.title(f'{feature} Distribution')
plt.tight_layout()
plt.show()

In [18]:
df = pd.read_csv("pump_data (2).csv")
print(df[['Voltage', 'Current']].describe())
print("\nFirst 10 rows:")
print(df[['Voltage', 'Current', 'State']].head(10))

           Voltage      Current
count  5248.000000  5248.000000
mean      0.877481   124.067151
std       1.882555    73.419075
min       0.060000    12.560000
25%       0.870000    43.127500
50%       0.880000   142.185000
75%       0.880000   170.110000
max     137.030000   432.660000

First 10 rows:
   Voltage  Current    State
0     0.88    44.10  DRY_RUN
1     0.96    36.56  DRY_RUN
2     0.84    34.09  DRY_RUN
3     0.81    35.21  DRY_RUN
4     0.90    36.61  DRY_RUN
5     0.88    41.70  DRY_RUN
6     0.88    38.33  DRY_RUN
7     0.89    40.73  DRY_RUN
8     0.88    32.41  DRY_RUN
9     0.87    44.47  DRY_RUN


In [13]:
import pandas as pd
df_check = pd.read_csv("pump_data (2).csv")
print(df_check['State'].value_counts())
print("\n")
print(df_check.describe())

State
DRY_RUN       1889
CAVITATION    1835
NORMAL        1524
Name: count, dtype: int64


              Time      Voltage      Current
count  5248.000000  5248.000000  5248.000000
mean    836.576006     0.877481   124.067151
std     567.434505     1.882555    73.419075
min       0.000000     0.060000    12.560000
25%     358.000000     0.870000    43.127500
50%     739.500000     0.880000   142.185000
75%    1284.250000     0.880000   170.110000
max    2045.000000   137.030000   432.660000


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout
import joblib

# Load data
df = pd.read_csv("pump_data (2).csv")
df["Current"] = pd.to_numeric(df["Current"], errors="coerce")
df = df.dropna().drop_duplicates()

if "Time" in df.columns:
    df = df.drop(columns=["Time"])
if "Voltage" in df.columns:
    df = df.drop(columns=["Voltage"])

print("Classes:\n", df["State"].value_counts())
print("\nCurrent stats:\n", df["Current"].describe())

# Encode labels
label_encoder = LabelEncoder()
df["State"] = label_encoder.fit_transform(df["State"])
print("\nClass mapping:", dict(zip(label_encoder.classes_,
                                   label_encoder.transform(label_encoder.classes_))))

# Split per class chronologically
train_list, test_list = [], []
for cls in df["State"].unique():
    cls_df = df[df["State"] == cls].copy()
    split  = int(len(cls_df) * 0.80)
    train_list.append(cls_df.iloc[:split])
    test_list.append(cls_df.iloc[split:])

train_df = pd.concat(train_list)
test_df  = pd.concat(test_list)

# Scale on Current only
feature_cols = ["Current"]
scaler = StandardScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
test_df[feature_cols]  = scaler.transform(test_df[feature_cols])

print("\nScaler mean:", scaler.mean_)
print("Scaler std: ", scaler.scale_)

np.save("pump_norm_mean.npy", scaler.mean_)
np.save("pump_norm_std.npy",  scaler.scale_)

# Window
def create_windows(X, y, window_size, step_size):
    Xw, yw = [], []
    for i in range(0, len(X) - window_size + 1, step_size):
        Xw.append(X[i:i+window_size])
        labels = y[i:i+window_size]
        unique, counts = np.unique(labels, return_counts=True)
        yw.append(unique[np.argmax(counts)])
    return np.array(Xw), np.array(yw)

WINDOW_SIZE = 50
STEP_SIZE   = 10

X_train, y_train = create_windows(
    train_df[feature_cols].values, train_df["State"].values, WINDOW_SIZE, STEP_SIZE)
X_test, y_test   = create_windows(
    test_df[feature_cols].values,  test_df["State"].values,  WINDOW_SIZE, STEP_SIZE)

print(f"\nX_train: {X_train.shape}")  # should be (samples, 50, 1)
print(f"X_test:  {X_test.shape}")

# Build model — input shape (50, 1)
NUM_CLASSES = len(label_encoder.classes_)

model = Sequential([
    Conv1D(filters=16, kernel_size=3, activation="relu",
           input_shape=(WINDOW_SIZE, 1)),
    MaxPooling1D(pool_size=2),
    LSTM(16, return_sequences=False),
    Dropout(0.2),
    Dense(NUM_CLASSES, activation="softmax")
])

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=30,
    batch_size=16,
    verbose=1
)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nFinal Test Accuracy: {test_acc * 100:.2f}%")

print("\nClassification Report:")
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
print(classification_report(y_test, y_pred,
                             target_names=label_encoder.classes_))

# Save
model.save("esp_model.h5")
joblib.dump(label_encoder, "esp_label_encoder.pkl")
print("\nSaved: esp_model.h5, esp_label_encoder.pkl")
print("Saved: pump_norm_mean.npy, pump_norm_std.npy")


Classes:
 State
DRY_RUN       1889
CAVITATION    1835
NORMAL        1524
Name: count, dtype: int64

Current stats:
 count    5248.000000
mean      124.067151
std        73.419075
min        12.560000
25%        43.127500
50%       142.185000
75%       170.110000
max       432.660000
Name: Current, dtype: float64

Class mapping: {'CAVITATION': 0, 'DRY_RUN': 1, 'NORMAL': 2}

Scaler mean: [123.71837303]
Scaler std:  [71.51569204]

X_train: (415, 50, 1)
X_test:  (101, 50, 1)



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1d (Conv1D)             (None, 48, 16)            64        
                                                                 
 max_pooling1d (MaxPooling1  (None, 24, 16)            0         
 D)                                                              
                                                                 
 lstm (LSTM)                 (None, 16

c:\Users\GLORY\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(



Saved: esp_model.h5, esp_label_encoder.pkl
Saved: pump_norm_mean.npy, pump_norm_std.npy
